# S19 — Ablation Study\nVary one hyperparameter at a time, train 5 epochs each, compare val_dice.

## Setup

In [ ]:
import sys, json, subprocess
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Ablation Config

In [ ]:
OUT = Path.cwd().parent / 'results' / 'ablation'
OUT.mkdir(parents=True, exist_ok=True)

ABLATIONS = {
    'baseline':    ([], 'Pretrained MobileNetV2, FocalLoss, lr=1e-3, bs=8'),
    'no_pretrain': (['--no-pretrained'], 'Scratch init'),
    'lr_1e-4':     (['--lr', '1e-4'], 'Lower LR'),
    'lr_1e-2':     (['--lr', '1e-2'], 'Higher LR'),
    'bs_4':        (['--batch-size', '4'], 'Smaller batch'),
    'no_focal':    (['--no-focal'], 'BCE+Dice instead of Focal'),
    'pos_weight_5':(['--no-focal', '--pos-weight', '5'], 'BCE+Dice, pos_weight=5'),
}
print(f'{len(ABLATIONS)} ablations configured')

## Run All Ablations

In [ ]:
train_script = str(Path.cwd().parent / 'scripts' / 'train.py')
python = sys.executable
results = []
for name, (extra_args, desc) in ABLATIONS.items():
    cmd = [python, train_script, '--epochs', '5', '--seed', '42',
           '--output-dir', str(OUT / name)] + extra_args
    print(f'\n{"="*60}\n{name}: {desc}\n{" ".join(cmd)}')
    r = subprocess.run(cmd, capture_output=True, text=True)
    for line in (r.stdout + r.stderr).split('\n'):
        if 'Best val dice' in line:
            dice = float(line.split(':')[-1].strip())
            results.append({'name': name, 'val_dice': dice})
            print(f'  -> Best val_dice: {dice:.4f}')
    if r.returncode != 0:
        print(f'  FAILED (exit {r.returncode})')
        print(r.stderr[-500:])

## Results Summary

In [ ]:
import pandas as pd
sorted_results = sorted(results, key=lambda x: x['val_dice'], reverse=True)
df = pd.DataFrame(sorted_results)
print(df.to_string(index=False))

with open(OUT / 'summary.json', 'w') as f:
    json.dump(sorted_results, f, indent=2)
print(f'\nSaved to {OUT / "summary.json"}')